
# Food.com Recipe Ranking System
## 02 – Data Cleaning and Preprocessing

### 1. Introduction

This notebook performs data cleaning and preprocessing on the original Food.com Recipes and User Interactions datasets.

The purpose of this stage is to improve data quality, handle missing and invalid values, convert columns into appropriate data types, and prepare the datasets for exploratory data analysis and recipe ranking.

### 2. Objectives

- Load the original datasets.
- Handle missing and invalid recipe information.
- Convert date columns into datetime format.
- Process nutrition information into individual numerical columns.
- Convert list-like columns into Python lists.
- Create additional recipe features.
- Clean and validate user interaction data.
- Save and reload the processed datasets.
- Verify the correctness of preprocessing operations.

### 3. Preprocessing Approach

Reusable Python functions developed in the `src/` directory will be imported and used throughout this notebook.

This approach avoids duplicating code and ensures that the same preprocessing logic can be reused in the Jupyter notebooks and the Streamlit application.

### 4. Data Integrity

The original CSV files will remain unchanged. All preprocessing operations will be performed on separate DataFrames, and the processed datasets will be stored separately.


In [39]:

# Import required libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Required libraries imported successfully!")


Required libraries imported successfully!


In [40]:

# Check the current working directory

import os

print("Current working directory:")
print(os.getcwd())


Current working directory:
/Users/mrmac/Documents/Venudatta_Python_project_My-recipe/notebooks


In [41]:

# Add the project root directory to Python's module search path

import sys

project_root = os.path.abspath("..")

if project_root not in sys.path:
    sys.path.insert(0, project_root)

print("Project root:", project_root)
print("Project root added to Python path:", project_root in sys.path)


Project root: /Users/mrmac/Documents/Venudatta_Python_project_My-recipe
Project root added to Python path: True


In [42]:
# Import reusable data preprocessing functions from src

from src.data_processing import (
    load_recipe_data,
    clean_recipe_data,
    process_nutrition_data,
    process_list_columns,
    create_recipe_features,
    save_processed_data,
    load_processed_data,
    clean_interaction_data,
    
)

print("All preprocessing functions imported successfully!")

All preprocessing functions imported successfully!


In [43]:
# Verify that all imported preprocessing functions are callable

preprocessing_functions = {
    "load_recipe_data": load_recipe_data,
    "clean_recipe_data": clean_recipe_data,
    "process_nutrition_data": process_nutrition_data,
    "process_list_columns": process_list_columns,
    "create_recipe_features": create_recipe_features,
    "save_processed_data": save_processed_data,
    "load_processed_data": load_processed_data
}

for function_name, function in preprocessing_functions.items():
    print(f"{function_name}: {callable(function)}")

load_recipe_data: True
clean_recipe_data: True
process_nutrition_data: True
process_list_columns: True
create_recipe_features: True
save_processed_data: True
load_processed_data: True


## 4. Load Raw Recipe Dataset

In this section, we load the original Food.com recipe dataset using
the reusable `load_recipe_data()` function.

The raw dataset is preserved without modifying the original CSV file.
All cleaning and transformations will be performed on a separate
working DataFrame.

In [44]:
# Define the raw recipe dataset path

recipes_path = os.path.join(
    project_root,
    "data",
    "raw",
    "RAW_recipes.csv"
)

# Load the raw recipe dataset

recipes = load_recipe_data(recipes_path)

print("Raw recipe dataset loaded successfully!")
print("Dataset shape:", recipes.shape)
print("\nFirst five records:")
display(recipes.head())

Raw recipe dataset loaded successfully!
Dataset shape: (231637, 12)

First five records:


,name,id,minutes,contributor_id,submitted,tags,nutrition,n_steps,steps,description,ingredients,n_ingredients
0,arriba baked winter squash mexican style,137739,55,47892,2005-09-16,"['60-minutes-or-less', 'time-to-make', 'course...","[51.5, 0.0, 13.0, 0.0, 2.0, 0.0, 4.0]",11,"['make a choice and proceed with recipe', 'dep...",autumn is my favorite time of year to cook! th...,"['winter squash', 'mexican seasoning', 'mixed ...",7
1,a bit different breakfast pizza,31490,30,26278,2002-06-17,"['30-minutes-or-less', 'time-to-make', 'course...","[173.4, 18.0, 0.0, 17.0, 22.0, 35.0, 1.0]",9,"['preheat oven to 425 degrees f', 'press dough...",this recipe calls for the crust to be prebaked...,"['prepared pizza crust', 'sausage patty', 'egg...",6
2,all in the kitchen chili,112140,130,196586,2005-02-25,"['time-to-make', 'course', 'preparation', 'mai...","[269.8, 22.0, 32.0, 48.0, 39.0, 27.0, 5.0]",6,"['brown ground beef in large pot', 'add choppe...",this modified version of 'mom's' chili was a h...,"['ground beef', 'yellow onions', 'diced tomato...",13
3,alouette potatoes,59389,45,68585,2003-04-14,"['60-minutes-or-less', 'time-to-make', 'course...","[368.1, 17.0, 10.0, 2.0, 14.0, 8.0, 20.0]",11,['place potatoes in a large pot of lightly sal...,"this is a super easy, great tasting, make ahea...","['spreadable cheese with garlic and herbs', 'n...",11
4,amish tomato ketchup for canning,44061,190,41706,2002-10-25,"['weeknight', 'time-to-make', 'course', 'main-...","[352.9, 1.0, 337.0, 23.0, 3.0, 0.0, 28.0]",5,['mix all ingredients& boil for 2 1 / 2 hours ...,my dh's amish mother raised him on this recipe...,"['tomato juice', 'apple cider vinegar', 'sugar...",8


In [45]:
# Inspect column names and data types

print("COLUMN NAMES")
print("-" * 60)

print(recipes.columns.tolist())

print("\nDATA TYPES")
print("-" * 60)

print(recipes.dtypes)

COLUMN NAMES
------------------------------------------------------------
['name', 'id', 'minutes', 'contributor_id', 'submitted', 'tags', 'nutrition', 'n_steps', 'steps', 'description', 'ingredients', 'n_ingredients']

DATA TYPES
------------------------------------------------------------
name                str
id                int64
minutes           int64
contributor_id    int64
submitted           str
tags                str
nutrition           str
n_steps           int64
steps               str
description         str
ingredients         str
n_ingredients     int64
dtype: object


In [46]:
# Check missing values in the raw recipe dataset

missing_values = recipes.isnull().sum()

print("MISSING VALUES BEFORE CLEANING")
print("-" * 60)

print(missing_values)

print("\nTOTAL MISSING VALUES:", missing_values.sum())

MISSING VALUES BEFORE CLEANING
------------------------------------------------------------
name                 1
id                   0
minutes              0
contributor_id       0
submitted            0
tags                 0
nutrition            0
n_steps              0
steps                0
description       4979
ingredients          0
n_ingredients        0
dtype: int64

TOTAL MISSING VALUES: 4980


## 5. Recipe Data Cleaning

In this section, we apply the reusable `clean_recipe_data()` function
to clean the raw recipe dataset.

Cleaning operations include:
- Removing recipes with missing or blank names.
- Converting zero cooking times into missing values.
- Converting submission dates into datetime format.
- Resetting the DataFrame index.

Missing descriptions are retained because they are not essential
for recipe ranking.

In [47]:
# Apply recipe data cleaning

cleaned_recipes = clean_recipe_data(recipes)

print("RECIPE DATA CLEANING COMPLETED")
print("-" * 60)

print("Original dataset shape:", recipes.shape)
print("Cleaned dataset shape:", cleaned_recipes.shape)

print("Rows removed:", len(recipes) - len(cleaned_recipes))

RECIPE DATA CLEANING COMPLETED
------------------------------------------------------------
Original dataset shape: (231637, 12)
Cleaned dataset shape: (231636, 12)
Rows removed: 1


In [48]:
# Verify recipe cleaning results

print("CLEANING VERIFICATION")
print("-" * 60)

print("Missing recipe names:", cleaned_recipes["name"].isna().sum())

print("Missing descriptions:", cleaned_recipes["description"].isna().sum())

print("Zero cooking times:", (cleaned_recipes["minutes"] == 0).sum())

print("Missing cooking times:", cleaned_recipes["minutes"].isna().sum())

print("Negative cooking times:", (cleaned_recipes["minutes"] < 0).sum())

print("Submitted column data type:", cleaned_recipes["submitted"].dtype)

CLEANING VERIFICATION
------------------------------------------------------------
Missing recipe names: 0
Missing descriptions: 4979
Zero cooking times: 0
Missing cooking times: 1094
Negative cooking times: 0
Submitted column data type: datetime64[us]


## 6. Nutrition Data Processing

The original nutrition column contains seven nutritional values
stored as string representations of Python lists.

We use the reusable `process_nutrition_data()` function to:
- Parse the nutrition values.
- Separate the seven values into individual columns.
- Convert the extracted values into numeric data types.
- Remove the original nutrition column after extraction.

In [49]:
# Process nutrition information

nutrition_processed_recipes = process_nutrition_data(cleaned_recipes)

print("NUTRITION PROCESSING COMPLETED")
print("-" * 60)

print("Dataset shape:", nutrition_processed_recipes.shape)

print("\nNew nutrition columns:")

nutrition_columns = [
    "calories",
    "total_fat_pdv",
    "sugar_pdv",
    "sodium_pdv",
    "protein_pdv",
    "saturated_fat_pdv",
    "carbohydrates_pdv"
]

print(nutrition_processed_recipes[nutrition_columns].head())

NUTRITION PROCESSING COMPLETED
------------------------------------------------------------
Dataset shape: (231636, 18)

New nutrition columns:
   calories  total_fat_pdv  sugar_pdv  sodium_pdv  protein_pdv  \
0      51.5            0.0       13.0         0.0          2.0   
1     173.4           18.0        0.0        17.0         22.0   
2     269.8           22.0       32.0        48.0         39.0   
3     368.1           17.0       10.0         2.0         14.0   
4     352.9            1.0      337.0        23.0          3.0   

   saturated_fat_pdv  carbohydrates_pdv  
0                0.0                4.0  
1               35.0                1.0  
2               27.0                5.0  
3                8.0               20.0  
4                0.0               28.0  


In [50]:
# Verify nutrition column data types and missing values

print("NUTRITION DATA TYPE VERIFICATION")
print("-" * 60)

print(nutrition_processed_recipes[nutrition_columns].dtypes)

print("\nMISSING VALUES IN NUTRITION COLUMNS")
print("-" * 60)

print(nutrition_processed_recipes[nutrition_columns].isna().sum())

print("\nORIGINAL NUTRITION COLUMN EXISTS:",
      "nutrition" in nutrition_processed_recipes.columns)

NUTRITION DATA TYPE VERIFICATION
------------------------------------------------------------
calories             float64
total_fat_pdv        float64
sugar_pdv            float64
sodium_pdv           float64
protein_pdv          float64
saturated_fat_pdv    float64
carbohydrates_pdv    float64
dtype: object

MISSING VALUES IN NUTRITION COLUMNS
------------------------------------------------------------
calories             0
total_fat_pdv        0
sugar_pdv            0
sodium_pdv           0
protein_pdv          0
saturated_fat_pdv    0
carbohydrates_pdv    0
dtype: int64

ORIGINAL NUTRITION COLUMN EXISTS: False


## 7. List Column Processing

The tags, ingredients, and steps columns contain string
representations of Python lists.

We use the reusable `process_list_columns()` function
to convert these strings into actual Python lists.

This allows us to perform list operations and extract
useful recipe information during exploratory data analysis.

In [51]:
# Convert string representations into Python lists

list_processed_recipes = process_list_columns(
    nutrition_processed_recipes
)

print("LIST COLUMN PROCESSING COMPLETED")
print("-" * 60)

print("Dataset shape:", list_processed_recipes.shape)

print("\nFirst recipe tags:")
print(list_processed_recipes["tags"].iloc[0])

print("\nFirst recipe ingredients:")
print(list_processed_recipes["ingredients"].iloc[0])

print("\nFirst recipe cooking steps:")
print(list_processed_recipes["steps"].iloc[0])

LIST COLUMN PROCESSING COMPLETED
------------------------------------------------------------
Dataset shape: (231636, 18)

First recipe tags:
['60-minutes-or-less', 'time-to-make', 'course', 'main-ingredient', 'cuisine', 'preparation', 'occasion', 'north-american', 'side-dishes', 'vegetables', 'mexican', 'easy', 'fall', 'holiday-event', 'vegetarian', 'winter', 'dietary', 'christmas', 'seasonal', 'squash']

First recipe ingredients:
['winter squash', 'mexican seasoning', 'mixed spice', 'honey', 'butter', 'olive oil', 'salt']

First recipe cooking steps:
['make a choice and proceed with recipe', 'depending on size of squash , cut into half or fourths', 'remove seeds', 'for spicy squash , drizzle olive oil or melted butter over each cut squash piece', 'season with mexican seasoning mix ii', 'for sweet squash , drizzle melted honey , butter , grated piloncillo over each cut squash piece', 'season with sweet mexican spice mix', 'bake at 350 degrees , again depending on size , for 40 minutes

In [52]:
# Verify that list columns contain actual Python lists

list_columns = ["tags", "ingredients", "steps"]

print("LIST COLUMN DATA TYPE VERIFICATION")
print("-" * 60)

for column in list_columns:
    print(
        f"{column}:",
        type(list_processed_recipes[column].iloc[0])
    )

print("\nALL VALUES ARE PYTHON LISTS:")

for column in list_columns:
    print(
        f"{column}:",
        list_processed_recipes[column].apply(
            lambda value: isinstance(value, list)
        ).all()
    )

LIST COLUMN DATA TYPE VERIFICATION
------------------------------------------------------------
tags: <class 'list'>
ingredients: <class 'list'>
steps: <class 'list'>

ALL VALUES ARE PYTHON LISTS:
tags: True
ingredients: True
steps: True


## 8. Recipe Feature Engineering

In this section, we create three additional recipe features
using the reusable create_recipe_features() function.

The features are:

- ingredient_count: Number of ingredients in each recipe.
- tag_count: Number of tags associated with each recipe.
- step_count: Number of cooking instructions.

These features will help us understand recipe complexity
and characteristics during exploratory data analysis.

In [53]:
# Create additional recipe features

featured_recipes = create_recipe_features(
    list_processed_recipes
)

print("RECIPE FEATURE ENGINEERING COMPLETED")
print("-" * 60)

print("Dataset shape:", featured_recipes.shape)

print("\nNew feature columns:")
print(
    featured_recipes[
        ["ingredient_count", "tag_count", "step_count"]
    ].head()
)

RECIPE FEATURE ENGINEERING COMPLETED
------------------------------------------------------------
Dataset shape: (231636, 21)

New feature columns:
   ingredient_count  tag_count  step_count
0                 7         20          11
1                 6         20           9
2                13          9           6
3                11         30          11
4                 8         21           5


In [54]:
# Verify engineered recipe features

feature_columns = [
    "ingredient_count",
    "tag_count",
    "step_count"
]

print("FEATURE DATA TYPE VERIFICATION")
print("-" * 60)

print(featured_recipes[feature_columns].dtypes)

print("\nMISSING VALUES")
print("-" * 60)

print(featured_recipes[feature_columns].isna().sum())

print("\nNEGATIVE VALUES")
print("-" * 60)

print((featured_recipes[feature_columns] < 0).sum())

FEATURE DATA TYPE VERIFICATION
------------------------------------------------------------
ingredient_count    int64
tag_count           int64
step_count          int64
dtype: object

MISSING VALUES
------------------------------------------------------------
ingredient_count    0
tag_count           0
step_count          0
dtype: int64

NEGATIVE VALUES
------------------------------------------------------------
ingredient_count    0
tag_count           0
step_count          0
dtype: int64


## 9. Save Processed Recipe Dataset

After completing recipe cleaning, nutrition processing,
list conversion, and feature engineering, we save the
processed dataset as a CSV file.

The processed dataset is stored separately from the
original raw dataset.

This allows the processed data to be reused in later
analysis notebooks and the Streamlit application.

In [55]:
# Define the processed dataset output path

processed_data_path = os.path.join(
    project_root,
    "data",
    "processed",
    "processed_recipes.csv"
)

# Save the processed recipe dataset

save_processed_data(
    featured_recipes,
    processed_data_path
)

print("PROCESSED RECIPE DATA SAVED SUCCESSFULLY")
print("-" * 60)

print("Output path:", processed_data_path)
print("Dataset shape:", featured_recipes.shape)

Processed recipe data saved successfully: /Users/mrmac/Documents/Venudatta_Python_project_My-recipe/data/processed/processed_recipes.csv
PROCESSED RECIPE DATA SAVED SUCCESSFULLY
------------------------------------------------------------
Output path: /Users/mrmac/Documents/Venudatta_Python_project_My-recipe/data/processed/processed_recipes.csv
Dataset shape: (231636, 21)


### 9.1 Reload and Verify Processed Dataset

We reload the saved CSV file using the reusable
load_processed_data() function.

This function restores the Python list columns and
converts the submission date back into datetime format.

The reloaded dataset is then compared with the
original processed DataFrame.

In [56]:
# Reload the saved processed recipe dataset

reloaded_recipes = load_processed_data(processed_data_path)

print("PROCESSED DATA RELOADING COMPLETED")
print("-" * 60)

print("Reloaded dataset shape:", reloaded_recipes.shape)

print("\nData types of restored list columns:")

for column in ["tags", "ingredients", "steps"]:
    print(column, type(reloaded_recipes[column].iloc[0]))

print("\nSubmission date data type:")
print(reloaded_recipes["submitted"].dtype)

PROCESSED DATA RELOADING COMPLETED
------------------------------------------------------------
Reloaded dataset shape: (231636, 21)

Data types of restored list columns:
tags <class 'list'>
ingredients <class 'list'>
steps <class 'list'>

Submission date data type:
datetime64[us]


In [57]:
# Final data-integrity verification

print("FINAL DATA INTEGRITY VERIFICATION")
print("-" * 60)

print(
    "Dataset shape matches:",
    featured_recipes.shape == reloaded_recipes.shape
)

print(
    "Column names match:",
    featured_recipes.columns.tolist()
    == reloaded_recipes.columns.tolist()
)

print(
    "Recipe IDs match:",
    featured_recipes["id"].equals(reloaded_recipes["id"])
)

print(
    "Recipe names match:",
    featured_recipes["name"].equals(reloaded_recipes["name"])
)

print(
    "List columns restored correctly:",
    all(
        reloaded_recipes[column].apply(
            lambda value: isinstance(value, list)
        ).all()
        for column in ["tags", "ingredients", "steps"]
    )
)

print(
    "Engineered features match:",
    featured_recipes[
        ["ingredient_count", "tag_count", "step_count"]
    ].equals(
        reloaded_recipes[
            ["ingredient_count", "tag_count", "step_count"]
        ]
    )
)

FINAL DATA INTEGRITY VERIFICATION
------------------------------------------------------------
Dataset shape matches: True
Column names match: True
Recipe IDs match: True
Recipe names match: True
List columns restored correctly: True
Engineered features match: True


In [58]:
# Inspect available functions in the data processing module

import inspect
import src.data_processing as data_processing

print("AVAILABLE DATA PROCESSING FUNCTIONS")
print("-" * 60)

for function_name, function in inspect.getmembers(
    data_processing,
    inspect.isfunction
):
    if function.__module__ == data_processing.__name__:
        print(function_name)

AVAILABLE DATA PROCESSING FUNCTIONS
------------------------------------------------------------
clean_interaction_data
clean_recipe_data
create_recipe_features
load_processed_data
load_recipe_data
process_list_columns
process_nutrition_data
save_processed_data


## 10. User Interaction Data Preprocessing

The Food.com interaction dataset contains user ratings and
written reviews associated with recipes.

In this section, we will:
- Load the original interaction dataset.
- Inspect its structure and data types.
- Convert interaction dates into datetime format.
- Preserve all original rating and review records.
- Validate the processed interaction dataset.

Zero ratings will be retained in the processed data and
excluded only when calculating recipe ranking statistics.

In [59]:
# Define the raw interaction dataset path

interactions_path = os.path.join(
    project_root,
    "data",
    "raw",
    "RAW_interactions.csv"
)

# Load the raw interaction dataset for initial inspection

interactions = pd.read_csv(interactions_path)

print("RAW INTERACTION DATASET LOADED")
print("-" * 60)

print("Dataset shape:", interactions.shape)

print("\nFirst five records:")

display(interactions.head())

RAW INTERACTION DATASET LOADED
------------------------------------------------------------
Dataset shape: (1132367, 5)

First five records:


,user_id,recipe_id,date,rating,review
0,38094,40893,2003-02-17,4,Great with a salad. Cooked on top of stove for...
1,1293707,40893,2011-12-21,5,"So simple, so delicious! Great for chilly fall..."
2,8937,44394,2002-12-01,4,This worked very well and is EASY. I used not...
3,126440,85009,2010-02-27,5,I made the Mexican topping and took it to bunk...
4,57222,85009,2011-10-01,5,"Made the cheddar bacon topping, adding a sprin..."


In [60]:
# Inspect interaction dataset data types

print("INTERACTION DATA TYPES")
print("-" * 60)

print(interactions.dtypes)


# Inspect missing values

print("\nMISSING VALUES")
print("-" * 60)

missing_values = interactions.isnull().sum()

missing_summary = pd.DataFrame({
    "Missing Values": missing_values,
    "Missing Percentage": (
        missing_values / len(interactions) * 100
    ).round(2)
})

display(missing_summary)

INTERACTION DATA TYPES
------------------------------------------------------------
user_id      int64
recipe_id    int64
date           str
rating       int64
review         str
dtype: object

MISSING VALUES
------------------------------------------------------------


,Missing Values,Missing Percentage
user_id,0,0.00
recipe_id,0,0.00
date,0,0.00
rating,0,0.00
review,169,0.01


In [61]:
# Clean the user interaction dataset

cleaned_interactions = clean_interaction_data(interactions)

print("INTERACTION DATA CLEANING COMPLETED")
print("-" * 60)

print("Original dataset shape:", interactions.shape)
print("Cleaned dataset shape:", cleaned_interactions.shape)

print("\nUpdated data types:")
print(cleaned_interactions.dtypes)

print("\nMissing values after cleaning:")
print(cleaned_interactions.isnull().sum())

INTERACTION DATA CLEANING COMPLETED
------------------------------------------------------------
Original dataset shape: (1132367, 5)
Cleaned dataset shape: (1132367, 5)

Updated data types:
user_id               int64
recipe_id             int64
date         datetime64[us]
rating                int64
review                  str
dtype: object

Missing values after cleaning:
user_id        0
recipe_id      0
date           0
rating         0
review       169
dtype: int64


In [62]:
# Verify rating values after interaction cleaning

print("RATING VALIDATION")
print("-" * 60)

print("Unique rating values:")
print(sorted(cleaned_interactions["rating"].unique()))

print("\nMinimum rating:",
      cleaned_interactions["rating"].min())

print("Maximum rating:",
      cleaned_interactions["rating"].max())

print("\nNumber of zero ratings:",
      (cleaned_interactions["rating"] == 0).sum())

print("\nNumber of ratings outside the 0-5 range:",
      (~cleaned_interactions["rating"].between(0, 5)).sum())

RATING VALIDATION
------------------------------------------------------------
Unique rating values:
[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]

Minimum rating: 0
Maximum rating: 5

Number of zero ratings: 60847

Number of ratings outside the 0-5 range: 0


In [63]:
# Verify original interaction data remains unchanged

print("ORIGINAL DATA PRESERVATION CHECK")
print("-" * 60)

print(
    "Original date data type:",
    interactions["date"].dtype
)

print(
    "Cleaned date data type:",
    cleaned_interactions["date"].dtype
)

print(
    "\nOriginal dataset shape unchanged:",
    interactions.shape == cleaned_interactions.shape
)

print(
    "Original date column remains string:",
    interactions["date"].dtype == "str"
)

print(
    "Cleaned date column converted to datetime:",
    pd.api.types.is_datetime64_any_dtype(
        cleaned_interactions["date"]
    )
)

ORIGINAL DATA PRESERVATION CHECK
------------------------------------------------------------
Original date data type: str
Cleaned date data type: datetime64[us]

Original dataset shape unchanged: True
Original date column remains string: True
Cleaned date column converted to datetime: True


## Conclusion

In this notebook, the Food.com recipe and user interaction datasets were cleaned and preprocessed using reusable functions from `src/data_processing.py`.

The recipe dataset was cleaned, nutrition values were extracted into separate numerical columns, list-based columns were converted into Python lists, and additional recipe features were created. The processed recipe dataset was saved and reloaded successfully, with data integrity verified.

The user interaction dataset was cleaned by converting interaction dates into datetime format. All original interaction records were preserved, including zero ratings and missing reviews. Rating values were also validated to ensure they remained within the expected range of 0–5.

All major preprocessing steps were successfully executed and verified.

The processed recipe data and cleaned interaction data are now ready for subsequent exploratory data analysis and visualization.